# Form Field Data Validation

This notebook validates common form fields:
- Names
- Phone numbers
- Date of birth
- Email
- Payment card info (Visa/Mastercard + Luhn, optional expiry/CVV)
- Permit purchased
- Address (street, province/state, postal code, country)

Outputs:
- Row-level validation errors
- Field-level error summary
- Cleaned/standardized columns

In [ ]:
# Great Expectations validation
!pip -q install great-expectations

import re
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datetime import datetime, date
from typing import Dict, List, Any, Optional, Tuple

## Load Data

Update the path/filename and column names to match your dataset.
You can load from:
- Kaggle dataset attached to the notebook
- A CSV you upload to the notebook

In [ ]:
# Example: Replace with your Kaggle input path
# df = pd.read_csv("/kaggle/input/your-dataset/forms.csv")

# Demo template (remove if you already have df):
df = pd.DataFrame({
    "first_name": ["Rita", "John3", "   ", None],
    "last_name": ["Sharma", "Doe", "O'Neil", "Ng"],
    "phone": ["+14035551234", "403-555-1234", "123", None],
    "dob": ["1988-09-09", "2030-01-01", "01/31/2010", "1999-02-29"],
    "email": ["rita@example.com", "bad@@mail.com", "a@b", None],
    "card_number": ["4111111111111111", "5555555555554444", "1234567890123456", None],
    "card_expiry": ["12/30", "01/20", "13/25", None],
    "card_cvv": ["123", "12a", "1234", None],
    "permit_purchased": ["Annual Park Vehicle", "UnknownPermit", "Fishing", None],
    "street": ["33-15075 60 Ave", "PO Box 123", "###", None],
    "city": ["Surrey", "Edmonton", "", "Calgary"],
    "province": ["BC", "AB", "Alberta", "XX"],
    "postal_code": ["V3S 1S1", "T5J0N3", "12345", None],
    "country": ["CA", "Canada", "US", None],
})

df.head()

## Configuration

Adjust rules here:
- Age range (min/max)
- Allowed countries / provinces
- Permit allowed list
- Required fields
- Whether to allow PO Box, etc.

In [ ]:
CONFIG = {
    # Age rules for DOB validation
    "min_age": 12,     # adjust as needed
    "max_age": 120,

    # Required fields
    "required_fields": [
        "first_name", "last_name", "phone", "dob", "email",
        "permit_purchased", "street", "city", "province", "postal_code", "country"
    ],

    # Permits (example list – replace with your real permit names/codes)
    "allowed_permits": {
        "Annual Park Vehicle",
        "Fishing",
        "Hunting",
        "Wildlife",
        "Parks",
        "Forestry"
    },

    # Country normalization & allowlist
    "country_aliases": {
        "CANADA": "CA",
        "CA": "CA",
        "UNITED STATES": "US",
        "USA": "US",
        "US": "US",
    },
    "allowed_countries": {"CA", "US"},

    # Canada provinces/territories (2-letter)
    "ca_provinces": {
        "AB","BC","MB","NB","NL","NS","NT","NU","ON","PE","QC","SK","YT"
    },

    # US states (2-letter)
    "us_states": {
        "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
        "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
        "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
    },

    # Address rules
    "allow_po_box": True,
}

## Helper Functions

Includes:
- string cleaning
- regex checks
- Luhn algorithm for card validation
- email/phone parsing rules
- postal code validation (Canada/US)

In [ ]:
def is_blank(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    return str(x).strip() == ""

def clean_str(x: Any) -> Optional[str]:
    if is_blank(x):
        return None
    return str(x).strip()

def normalize_spaces(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    return re.sub(r"\s+", " ", s).strip()

def only_digits(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    digits = re.sub(r"\D", "", s)
    return digits if digits else None

def normalize_country(country: Any) -> Optional[str]:
    c = clean_str(country)
    if c is None:
        return None
    c_up = re.sub(r"\s+", " ", c).strip().upper()
    return CONFIG["country_aliases"].get(c_up, c_up)

# -----------------------
# Name validation
# -----------------------
NAME_ALLOWED = re.compile(r"^[A-Za-z][A-Za-z\-\.\' ]*[A-Za-z]$")  # letters + - . ' space, no leading/trailing non-letter
def validate_name(value: Any, field_label: str, min_len: int = 1, max_len: int = 80) -> List[str]:
    errs = []
    v = normalize_spaces(clean_str(value))
    if v is None:
        return [f"{field_label}: required"]  # can be changed if optional
    if len(v) < min_len:
        errs.append(f"{field_label}: too short (<{min_len})")
    if len(v) > max_len:
        errs.append(f"{field_label}: too long (>{max_len})")
    if not NAME_ALLOWED.match(v):
        errs.append(f"{field_label}: invalid characters/format")
    # disallow all same char e.g., "AAAA"
    if len(set(v.replace(" ", "").lower())) == 1 and len(v.replace(" ", "")) >= 4:
        errs.append(f"{field_label}: suspicious repeated characters")
    return errs

# -----------------------
# Email validation (practical)
# -----------------------
EMAIL_RE = re.compile(
    r"^[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@"
    r"[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)+$"
)

def validate_email(value: Any) -> List[str]:
    v = clean_str(value)
    if v is None:
        return ["email: required"]
    v = v.strip()
    errs = []
    if len(v) > 254:
        errs.append("email: too long (>254)")
    if not EMAIL_RE.match(v):
        errs.append("email: invalid format")
    # basic domain sanity
    if "@" in v:
        local, domain = v.rsplit("@", 1)
        if len(local) == 0 or len(domain) == 0:
            errs.append("email: missing local/domain")
        if domain.startswith(".") or domain.endswith("."):
            errs.append("email: invalid domain dots")
    return errs

# -----------------------
# Phone validation
# -----------------------
E164_RE = re.compile(r"^\+[1-9]\d{7,14}$")  # E.164 length 8-15 total digits

def normalize_phone(value: Any) -> Optional[str]:
    v = clean_str(value)
    if v is None:
        return None
    v = v.strip()
    # if user typed +... keep it, else strip to digits
    if v.startswith("+"):
        digits = "+" + re.sub(r"\D", "", v)
        return digits if digits != "+" else None
    digits = only_digits(v)
    return digits

def validate_phone(value: Any, default_country: str = "CA") -> List[str]:
    """
    - Accept E.164 (preferred)
    - Also accept CA/US 10-digit or 11-digit starting with 1
    """
    v = clean_str(value)
    if v is None:
        return ["phone: required"]

    errs = []
    norm = normalize_phone(v)
    if norm is None:
        return ["phone: empty after cleaning"]

    if norm.startswith("+"):
        if not E164_RE.match(norm):
            errs.append("phone: invalid E.164 format")
        return errs

    # non +: treat as national digits
    digits = norm
    if default_country in {"CA", "US"}:
        if len(digits) == 10:
            # NANP: NXXNXXXXXX - first digit of area/exchange cannot be 0/1
            if digits[0] in "01" or digits[3] in "01":
                errs.append("phone: invalid NANP area/exchange code")
        elif len(digits) == 11 and digits.startswith("1"):
            if digits[1] in "01" or digits[4] in "01":
                errs.append("phone: invalid NANP area/exchange code")
        else:
            errs.append("phone: invalid length for CA/US (need 10 digits or 11 starting with 1)")
    else:
        # generic fallback
        if len(digits) < 8 or len(digits) > 15:
            errs.append("phone: invalid length (8-15 digits expected)")
    return errs

# -----------------------
# DOB validation
# -----------------------
def parse_dob(value: Any) -> Tuple[Optional[date], Optional[str]]:
    """
    Supports common formats:
    - YYYY-MM-DD
    - MM/DD/YYYY
    - DD/MM/YYYY (ambiguous; we try MM/DD first unless it fails)
    """
    v = clean_str(value)
    if v is None:
        return None, "dob: required"

    v = v.strip()
    fmts = ["%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y"]
    for fmt in fmts:
        try:
            d = datetime.strptime(v, fmt).date()
            return d, None
        except ValueError:
            pass
    return None, "dob: invalid date format"

def calc_age(dob: date, today: Optional[date] = None) -> int:
    if today is None:
        today = date.today()
    years = today.year - dob.year
    if (today.month, today.day) < (dob.month, dob.day):
        years -= 1
    return years

def validate_dob(value: Any, min_age: int, max_age: int) -> List[str]:
    dob, err = parse_dob(value)
    if err:
        return [err]
    errs = []
    today = date.today()
    if dob > today:
        errs.append("dob: cannot be in the future")
        return errs
    age = calc_age(dob, today=today)
    if age < min_age:
        errs.append(f"dob: age {age} < min_age {min_age}")
    if age > max_age:
        errs.append(f"dob: age {age} > max_age {max_age}")
    return errs

# -----------------------
# Card validation
# -----------------------
def luhn_check(num: str) -> bool:
    # num must be digits only
    total = 0
    rev = num[::-1]
    for i, ch in enumerate(rev):
        d = ord(ch) - 48
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def detect_card_scheme(digits: str) -> Optional[str]:
    """
    Minimal scheme detection (not exhaustive):
    Visa: starts 4, length 13/16/19
    Mastercard: 51-55 or 2221-2720, length 16
    """
    if digits.startswith("4") and len(digits) in {13, 16, 19}:
        return "VISA"
    if len(digits) == 16:
        prefix2 = int(digits[:2])
        prefix4 = int(digits[:4])
        if 51 <= prefix2 <= 55:
            return "MASTERCARD"
        if 2221 <= prefix4 <= 2720:
            return "MASTERCARD"
    return None

EXP_RE = re.compile(r"^(0[1-9]|1[0-2])\/?(\d{2}|\d{4})$")

def validate_card_number(value: Any) -> List[str]:
    v = clean_str(value)
    if v is None:
        # often optional unless payment required; keep as required here if payment present
        return ["card_number: required"]
    digits = only_digits(v)
    if digits is None:
        return ["card_number: no digits found"]

    errs = []
    if len(digits) < 12 or len(digits) > 19:
        errs.append("card_number: invalid length (12-19 digits)")
        return errs

    scheme = detect_card_scheme(digits)
    if scheme is None:
        errs.append("card_number: unknown/unsupported scheme (expected Visa/Mastercard)")
    if not luhn_check(digits):
        errs.append("card_number: failed Luhn check")
    return errs

def validate_card_expiry(value: Any) -> List[str]:
    v = clean_str(value)
    if v is None:
        return []  # optional
    v = v.replace(" ", "")
    m = EXP_RE.match(v)
    if not m:
        return ["card_expiry: invalid format (MM/YY or MM/YYYY)"]
    mm = int(m.group(1))
    yy = m.group(2)
    year = int(yy) + (2000 if len(yy) == 2 else 0)
    # expiry end-of-month check
    today = date.today()
    # treat as last day of month; approximate by comparing month/year
    if (year, mm) < (today.year, today.month):
        return ["card_expiry: expired"]
    return []

def validate_cvv(value: Any, scheme: Optional[str] = None) -> List[str]:
    v = clean_str(value)
    if v is None:
        return []  # optional
    digits = only_digits(v)
    if digits is None:
        return ["card_cvv: must be digits"]
    if scheme == "AMEX":
        if len(digits) != 4:
            return ["card_cvv: AMEX requires 4 digits"]
    else:
        if len(digits) != 3:
            return ["card_cvv: requires 3 digits"]
    return []

# -----------------------
# Permit validation
# -----------------------
def validate_permit(value: Any, allowed: set) -> List[str]:
    v = normalize_spaces(clean_str(value))
    if v is None:
        return ["permit_purchased: required"]
    if v not in allowed:
        return ["permit_purchased: not in allowed list"]
    return []

# -----------------------
# Address validation
# -----------------------
POSTAL_CA_RE = re.compile(r"^[A-Za-z]\d[A-Za-z][ -]?\d[A-Za-z]\d$")  # A1A 1A1
ZIP_US_RE = re.compile(r"^\d{5}(-\d{4})?$")

def normalize_province(value: Any) -> Optional[str]:
    v = normalize_spaces(clean_str(value))
    if v is None:
        return None
    return v.strip().upper()

def validate_street(value: Any, allow_po_box: bool = True) -> List[str]:
    v = normalize_spaces(clean_str(value))
    if v is None:
        return ["street: required"]
    errs = []
    if len(v) < 3:
        errs.append("street: too short")
    if len(v) > 120:
        errs.append("street: too long (>120)")
    # basic: must contain at least one alphanumeric
    if not re.search(r"[A-Za-z0-9]", v):
        errs.append("street: must include letters/numbers")
    if not allow_po_box:
        if re.search(r"\bP(\.|)\s*O(\.|)\s*BOX\b", v, flags=re.IGNORECASE):
            errs.append("street: PO Box not allowed")
    return errs

def validate_city(value: Any) -> List[str]:
    v = normalize_spaces(clean_str(value))
    if v is None:
        return ["city: required"]
    errs = []
    if len(v) < 2:
        errs.append("city: too short")
    if len(v) > 80:
        errs.append("city: too long (>80)")
    if not re.match(r"^[A-Za-z][A-Za-z\-\.' ]*[A-Za-z]$", v):
        errs.append("city: invalid characters/format")
    return errs

def validate_country(value: Any) -> Tuple[Optional[str], List[str]]:
    norm = normalize_country(value)
    if norm is None:
        return None, ["country: required"]
    errs = []
    if norm not in CONFIG["allowed_countries"]:
        errs.append("country: not allowed")
    return norm, errs

def validate_province(value: Any, country_norm: Optional[str]) -> List[str]:
    v = normalize_province(value)
    if v is None:
        return ["province: required"]
    errs = []
    if country_norm == "CA":
        # allow full name? you can add mappings if needed
        if v not in CONFIG["ca_provinces"]:
            errs.append("province: invalid CA province code (expected 2-letter)")
    elif country_norm == "US":
        if v not in CONFIG["us_states"]:
            errs.append("province: invalid US state code (expected 2-letter)")
    else:
        # If country not allowed, we still validate basic shape
        if len(v) < 2:
            errs.append("province: too short")
    return errs

def validate_postal(value: Any, country_norm: Optional[str]) -> List[str]:
    v = normalize_spaces(clean_str(value))
    if v is None:
        return ["postal_code: required"]
    v = v.strip()
    errs = []
    if country_norm == "CA":
        if not POSTAL_CA_RE.match(v):
            errs.append("postal_code: invalid Canadian postal format (A1A 1A1)")
    elif country_norm == "US":
        if not ZIP_US_RE.match(v):
            errs.append("postal_code: invalid US ZIP format (12345 or 12345-6789)")
    else:
        # generic
        if len(v) < 3 or len(v) > 12:
            errs.append("postal_code: invalid length")
    return errs

## Validation Runner

Creates:
- `errors_by_row` (list of messages per row)
- `invalid_rows_df` (only invalid rows with errors)
- `field_error_summary` (count by error type)
- Cleaned columns (normalized phone/country/province/postal)

In [ ]:
def validate_row(row: pd.Series) -> Dict[str, Any]:
    errors: List[str] = []

    # Required fields (presence)
    for f in CONFIG["required_fields"]:
        if f in row.index and is_blank(row[f]):
            errors.append(f"{f}: required")

    # Names
    errors += validate_name(row.get("first_name"), "first_name")
    errors += validate_name(row.get("last_name"), "last_name")

    # Phone
    errors += validate_phone(row.get("phone"), default_country="CA")

    # DOB
    errors += validate_dob(row.get("dob"), CONFIG["min_age"], CONFIG["max_age"])

    # Email
    errors += validate_email(row.get("email"))

    # Permit
    errors += validate_permit(row.get("permit_purchased"), CONFIG["allowed_permits"])

    # Country/Province/Postal
    country_norm, country_errs = validate_country(row.get("country"))
    errors += country_errs
    errors += validate_province(row.get("province"), country_norm)
    errors += validate_postal(row.get("postal_code"), country_norm)

    # Street/City
    errors += validate_street(row.get("street"), allow_po_box=CONFIG["allow_po_box"])
    errors += validate_city(row.get("city"))

    # Card (if any card fields provided, validate)
    card_any_present = any(not is_blank(row.get(c)) for c in ["card_number", "card_expiry", "card_cvv"])
    if card_any_present:
        # card number required if payment present
        cn = row.get("card_number")
        cn_digits = only_digits(clean_str(cn)) if not is_blank(cn) else None
        scheme = detect_card_scheme(cn_digits) if cn_digits else None

        errors += validate_card_number(cn)
        errors += validate_card_expiry(row.get("card_expiry"))
        errors += validate_cvv(row.get("card_cvv"), scheme=scheme)

    # Deduplicate while preserving order
    seen = set()
    errors_unique = []
    for e in errors:
        if e not in seen:
            seen.add(e)
            errors_unique.append(e)

    return {
        "is_valid": len(errors_unique) == 0,
        "errors": errors_unique,
        "country_norm": country_norm,
        "phone_norm": normalize_phone(row.get("phone")),
        "province_norm": normalize_province(row.get("province")),
        "postal_norm": normalize_spaces(clean_str(row.get("postal_code"))),
    }

results = df.apply(validate_row, axis=1, result_type="expand")
results.head()

## Attach Results & View Invalid Rows

In [ ]:
df_validated = df.copy()
df_validated["is_valid"] = results["is_valid"]
df_validated["errors"] = results["errors"]

# cleaned columns
df_validated["country_norm"] = results["country_norm"]
df_validated["phone_norm"] = results["phone_norm"]
df_validated["province_norm"] = results["province_norm"]
df_validated["postal_norm"] = results["postal_norm"]

invalid_rows_df = df_validated.loc[~df_validated["is_valid"]].copy()
valid_rows_df = df_validated.loc[df_validated["is_valid"]].copy()

print("Total rows:", len(df_validated))
print("Valid rows:", len(valid_rows_df))
print("Invalid rows:", len(invalid_rows_df))

invalid_rows_df.head(10)

## Field-Level Error Summary

Counts how often each error message appears.

In [ ]:
from collections import Counter

all_errors = []
for errs in df_validated["errors"]:
    all_errors.extend(errs)

error_counts = Counter(all_errors)
field_error_summary = pd.DataFrame(
    [{"error": k, "count": v} for k, v in error_counts.items()]
).sort_values(["count", "error"], ascending=[False, True])

field_error_summary.head(50)

## Save Outputs

Exports:
- validated_full.csv
- invalid_rows_only.csv
- error_summary.csv

In [ ]:
df_validated.to_csv("validated_full.csv", index=False)
invalid_rows_df.to_csv("invalid_rows_only.csv", index=False)
field_error_summary.to_csv("error_summary.csv", index=False)

print("Saved: validated_full.csv, invalid_rows_only.csv, error_summary.csv")

## Optional: Great Expectations (GE)

This adds a second validation layer with expectations.
Use it if you want a reusable validation suite/report.

Note: GE won't replace the more complex logic (like Luhn),
but it helps with standard checks and automated reporting.

In [ ]:
# OPTIONAL SECTION - run only if you installed great-expectations
# !pip -q install great-expectations

try:
    import great_expectations as ge

    ge_df = ge.from_pandas(df.copy())

    # Required non-null expectations
    for col in CONFIG["required_fields"]:
        if col in ge_df.columns:
            ge_df.expect_column_values_to_not_be_null(col)

    # Basic string length expectations (examples)
    if "first_name" in ge_df.columns:
        ge_df.expect_column_value_lengths_to_be_between("first_name", 1, 80)

    if "email" in ge_df.columns:
        ge_df.expect_column_values_to_match_regex("email", EMAIL_RE.pattern)

    if "postal_code" in ge_df.columns:
        # This is generic; country-specific checks are better in our custom runner
        ge_df.expect_column_values_to_not_be_null("postal_code")

    ge_report = ge_df.validate()
    print("GE success:", ge_report["success"])
except Exception as e:
    print("Great Expectations not run:", e)

In [ ]:
## How to Adapt

1) Rename columns in the config + validator to match your dataset.
2) Replace `allowed_permits` with your real permit list (codes or names).
3) If your system supports only Canada, set:
   - allowed_countries = {"CA"}
   - require province in ca_provinces
4) If you need stricter rules:
   - disallow PO Box
   - enforce specific phone format (E.164 only)
   - enforce DOB must be YYYY-MM-DD only
   - enforce card fields required only when permit requires payment